In [1]:
from pathlib import Path
import os, sys, json, shutil, subprocess, platform
from pprint import pprint

import numpy as np
import pandas as pd

In [2]:
## Confirm DIRs

DATA_ROOT = Path("/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768")
PROJECT_ROOT = DATA_ROOT / "rsCVR_project"
INDEX_DIR = PROJECT_ROOT / "index"
LOG_DIR = PROJECT_ROOT / "logs"
QC_DIR = PROJECT_ROOT / "qc"
RAPIDTIDE_DIR = PROJECT_ROOT / "rapidtide"
SPM_DIR = PROJECT_ROOT / "spm_preproc"
SUMMARY_DIR = PROJECT_ROOT / "summary"

for d in [PROJECT_ROOT, INDEX_DIR, LOG_DIR, QC_DIR, RAPIDTIDE_DIR, SPM_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT   :", DATA_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

DATA_ROOT   : /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768
PROJECT_ROOT: /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/rsCVR_project


In [11]:
## Confirm TOOLs

def run_cmd(cmd):
    try:
        out = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False
        )
        return {
            "returncode": out.returncode,
            "stdout": out.stdout.strip(),
            "stderr": out.stderr.strip()
        }
    except Exception as e:
        return {
            "returncode": -1,
            "stdout": "",
            "stderr": str(e)
        }
env_report = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "python_executable": sys.executable,
    "data_root_exists": DATA_ROOT.exists(),
    "jupyter_working_dir": str(Path.cwd()),
}
pprint(env_report)


# Confirm Python packages
pkg_report = {}

for pkg_name in ["numpy", "pandas", "matplotlib", "nibabel"]:
    try:
        __import__(pkg_name)
        pkg_report[pkg_name] = "OK"
    except Exception as e:
        pkg_report[pkg_name] = f"FAIL: {e}"
# spm-python 계열은 이름이 환경마다 다를 수 있어 여러 후보를 확인
for pkg_name in ["spm", "spm_python", "nipype"]:
    try:
        __import__(pkg_name)
        pkg_report[pkg_name] = "OK"
    except Exception as e:
        pkg_report[pkg_name] = f"FAIL: {e}"
pprint(pkg_report)


# Confirm CLI tools
cli_report = {}
for exe in ["rapidtide", "matlab", "fslhd", "bet", "python3"]:
    cli_report[exe] = shutil.which(exe)
pprint(cli_report)


# Confirm rapidtide version
if shutil.which("rapidtide"):
    rt = run_cmd(["rapidtide", "--help"])
    print("rapidtide callable:", rt["returncode"] == 0)
    print((rt["stdout"][:800] if rt["stdout"] else rt["stderr"][:800]))
else:
    print("rapidtide command not found in PATH")


# confirm MATLAB
if shutil.which("matlab"):
    ml = run_cmd(["matlab", "-batch", "disp(version)"])
    print("matlab callable:", ml["returncode"] == 0)
    print(ml["stdout"][:500] if ml["stdout"] else ml["stderr"][:500])
else:
    print("matlab command not found in PATH")


# Confirm SPM location (candidate)
spm_candidates = [
    Path.home() / "spm12",
    Path("/opt/spm12"),
    Path("/usr/local/spm12"),
    Path("/usr/local/MATLAB/spm12"),
]

spm_found = [str(p) for p in spm_candidates if p.exists()]
print("SPM candidate paths found:")
pprint(spm_found if spm_found else ["Not found"])

{'data_root_exists': True,
 'jupyter_working_dir': '/home/bami/Documents/JHA/rsCVR',
 'platform': 'Linux-5.15.0-139-generic-x86_64-with-glibc2.31',
 'python_executable': '/home/bami/anaconda3/envs/JHA-spmCVR/bin/python',
 'python_version': '3.10.20 | packaged by conda-forge | (main, Mar  5 2026, '
                   '16:42:22) [GCC 14.3.0]'}
{'matplotlib': 'OK',
 'nibabel': 'OK',
 'nipype': "FAIL: No module named 'nipype'",
 'numpy': 'OK',
 'pandas': 'OK',
 'spm': "FAIL: No module named 'spm'",
 'spm_python': "FAIL: No module named 'spm_python'"}
{'bet': '/home/bami/fsl/share/fsl/bin/bet',
 'fslhd': '/home/bami/fsl/share/fsl/bin/fslhd',
 'matlab': None,
 'python3': '/home/bami/anaconda3/bin/python3',
 'rapidtide': None}
rapidtide command not found in PATH
matlab command not found in PATH
SPM candidate paths found:
['Not found']


In [12]:
import sys
import site
import subprocess
from pathlib import Path

print("sys.executable:", sys.executable)
print("env bin exists:", Path(sys.executable).parent.exists())
print("site-packages:")
for p in site.getsitepackages():
    print(" -", p)

print("\nPATH:")
import os
print(os.environ.get("PATH", ""))

sys.executable: /home/bami/anaconda3/envs/JHA-spmCVR/bin/python
env bin exists: True
site-packages:
 - /home/bami/anaconda3/envs/JHA-spmCVR/lib/python3.10/site-packages

PATH:
/home/bami/anaconda3/bin:/home/bami/fsl/share/fsl/bin:/home/bami/fsl/share/fsl/bin:/home/bami/anaconda3/bin:/home/bami/anaconda3/condabin:/home/bami/anaconda3/condabin:/usr/local/cuda-11.2/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin


In [13]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "show", "rapidtide"], text=True)
subprocess.run([sys.executable, "-m", "pip", "show", "spm-python"], text=True)

Name: rapidtide
Version: 3.1.9
Summary: Tools for performing correlation analysis on fMRI data.
Home-page: https://github.com/bbfrederick/rapidtide
Author: Taylor Salo, Daniel M. Drucker, Ph.D., Jeffrey N Stout, Yaroslav O. Halchenko, Derek Monroe
Author-email: "Blaise deB. Frederick" <blaise.frederick@gmail.com>
License: Apache-2.0
Location: /home/bami/anaconda3/envs/JHA-spmCVR/lib/python3.10/site-packages
Requires: matplotlib, nibabel, nilearn, numpy, pandas, pyarrow, pyfftw, pyqt6, pyqt6-sip, pyqtgraph, pywavelets, requests, scikit-image, scikit-learn, scipy, statsmodels, torch, tqdm, versioneer
Required-by: 


CompletedProcess(args=['/home/bami/anaconda3/envs/JHA-spmCVR/bin/python', '-m', 'pip', 'show', 'spm-python'], returncode=1)

In [14]:
import sys, os, shutil, importlib.util
from pathlib import Path

ENV_BIN = Path(sys.executable).parent
os.environ["PATH"] = f"{ENV_BIN}:{os.environ['PATH']}"

print("Python:", sys.executable)
print("ENV_BIN:", ENV_BIN)
print("rapidtide exe exists:", (ENV_BIN / "rapidtide").exists())
print("which rapidtide:", shutil.which("rapidtide"))
print("spm spec:", importlib.util.find_spec("spm"))

Python: /home/bami/anaconda3/envs/JHA-spmCVR/bin/python
ENV_BIN: /home/bami/anaconda3/envs/JHA-spmCVR/bin
rapidtide exe exists: True
which rapidtide: /home/bami/anaconda3/envs/JHA-spmCVR/bin/rapidtide
spm spec: ModuleSpec(name='spm', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fe8e24d0f40>, origin='/home/bami/anaconda3/envs/JHA-spmCVR/lib/python3.10/site-packages/spm/__init__.py', submodule_search_locations=['/home/bami/anaconda3/envs/JHA-spmCVR/lib/python3.10/site-packages/spm'])


In [15]:
from spm import *
print("spm import OK")

spm import OK
